In [1]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
    
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics import accuracy_score, f1_score

from src.exploration import SOURCE, exporter_data
from src.Preprocessing import definir_xy, separer_types, cree_preprocessing
from src.train import create_models
from src.validation import cross_validate_models, grid_search_models

In [2]:
df = exporter_data(SOURCE)
df = df.drop_duplicates()
X, y = definir_xy(df, "Mental_Health_Risk")
print("X:", X.shape, "| y:", y.shape)

X: (2000, 15) | y: (2000,)


In [3]:
num_col, cat_col = separer_types(X)
preprocessor = cree_preprocessing(num_col, cat_col)
models = create_models(preprocessor)

variable numeriques: ['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']
variable categorielles: ['Gender', 'Physical_Activity', 'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History']


In [4]:
cv_df = cross_validate_models(models, X, y)
cv_df


 comparaison des models:                      Accuracy  Precision  Recall  F1-score  ROC-AUC
Model                                                              
Logistic Regression    0.7505     0.7524  0.7499    0.7504   0.9072
Random Forest          0.6775     0.6735  0.6765    0.6737   0.8532
Linear SVM             0.7270     0.7314  0.7264    0.7277   0.8866


,Accuracy,Precision,Recall,F1-score,ROC-AUC
Model,,,,,
Logistic Regression,0.7505,0.7524,0.7499,0.7504,0.9072
Random Forest,0.6775,0.6735,0.6765,0.6737,0.8532
Linear SVM,0.7270,0.7314,0.7264,0.7277,0.8866


In [5]:
best_models, gs_df = grid_search_models(models, X, y)
gs_df

Grid Search des modeles:

Optimisation: Logistic Regression
Fitting 5 folds for each of 10 candidates, totalling 50 fits
meilleurs parametres: {'model__C': 10, 'model__class_weight': 'balanced', 'model__solver': 'lbfgs'}
meilleur f1-score: 0.754

Optimisation: Random Forest
Fitting 5 folds for each of 36 candidates, totalling 180 fits
meilleurs parametres: {'model__class_weight': 'balanced', 'model__max_depth': 20, 'model__min_samples_split': 2, 'model__n_estimators': 100}
meilleur f1-score: 0.6886

Optimisation: Linear SVM
Fitting 5 folds for each of 24 candidates, totalling 120 fits


c:\Users\safiy\Desktop\ProjetsPython\mental_health_prediction\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


meilleurs parametres: {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__kernel': 'linear'}
meilleur f1-score: 0.7508

resultats Grid Search:
                                                       Best Parameters  \
Model                                                                    
Logistic Regression  {'model__C': 10, 'model__class_weight': 'balan...   
Random Forest        {'model__class_weight': 'balanced', 'model__ma...   
Linear SVM           {'model__C': 0.1, 'model__class_weight': 'bala...   

                     Best F1-score  
Model                               
Logistic Regression       0.754028  
Random Forest             0.688627  
Linear SVM                0.750830  


,Best Parameters,Best F1-score
Model,,
Logistic Regression,"{'model__C': 10, 'model__class_weight': 'balan...",0.754028
Random Forest,"{'model__class_weight': 'balanced', 'model__ma...",0.688627
Linear SVM,"{'model__C': 0.1, 'model__class_weight': 'bala...",0.750830


In [6]:
os.makedirs("../models", exist_ok=True)
for name, model in best_models.items():
    fname = name.replace(" ", "_").lower() + "_tuned.pkl"
    joblib.dump(model, f"../models/{fname}")
    print("sauvegarde:", fname)

sauvegarde: logistic_regression_tuned.pkl
sauvegarde: random_forest_tuned.pkl
sauvegarde: linear_svm_tuned.pkl


In [7]:
DATA_PATH = "../data/processed"
x_train = pd.read_csv(f"{DATA_PATH}/x_train.csv")
y_train = pd.read_csv(f"{DATA_PATH}/y_train.csv").squeeze("columns")
x_test  = pd.read_csv(f"{DATA_PATH}/x_test.csv")
y_test  = pd.read_csv(f"{DATA_PATH}/y_test.csv").squeeze("columns")

for name, model in best_models.items():
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    print(f"{name:25s} | acc={accuracy_score(y_test, pred):.4f} | f1_macro={f1_score(y_test, pred, average='macro'):.4f}")

Logistic Regression       | acc=0.7575 | f1_macro=0.7578
Random Forest             | acc=0.6975 | f1_macro=0.6976


c:\Users\safiy\Desktop\ProjetsPython\mental_health_prediction\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Linear SVM                | acc=0.7700 | f1_macro=0.7700


In [8]:
best_tuned_name = max(best_models, key=lambda n: f1_score(y_test, best_models[n].predict(x_test), average="macro"))
print("Meilleur modele optimise:", best_tuned_name)
os.makedirs("../models", exist_ok=True)
joblib.dump(best_models[best_tuned_name], "../models/best_model.pkl")
joblib.dump(best_tuned_name,               "../models/best_model_name.pkl")
print("Sauvegarde dans ../models/best_model.pkl")

Meilleur modele optimise: Linear SVM
Sauvegarde dans ../models/best_model.pkl
